# 03 — Baseline ML Models: Ridge & Lasso

This notebook establishes **baseline predictive models** for next-month factor returns using
Ridge and Lasso regression. These serve as a reference point for the more sophisticated
walk-forward timing models in notebook 04.

**Design choices:**
- Time-series train/validation/test splits only — no shuffling
- Train-only imputation and scaling (no data leakage)
- Shared feature set across all five factors
- Primary metric: **Information Coefficient (IC)** — rank correlation of predicted vs actual returns

**Splits:**
- Train: 2000-03 → 2015-12 (190 months)
- Validation: 2016-01 → 2019-12 (48 months) — used for hyperparameter tuning
- Test: 2020-01 → 2025-11 (73 months) — held out entirely until final evaluation

**Sections:**
1. Setup & Feature Engineering
2. Ridge Regression
3. Lasso Regression
4. Classification Targets (Direction / Outperformance)
5. Summary

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.metrics import mean_squared_error, r2_score

from utils import information_coefficient

warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 10})

## 1. Setup & Feature Engineering

In [ ]:
df = pd.read_csv('data/FinalMonthlyDataset_ours_ff_macro.csv')
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date').reset_index(drop=True)

# Target configuration: predict net factor returns 1 month ahead
HORIZON_MONTHS = 1
factor_cols = {
    'value':      'fac_value_net',
    'momentum':   'fac_momentum_net',
    'quality':    'fac_quality_net',
    'investment': 'fac_investment_net',
    'size':       'fac_size_net',
}
target_cols = {name: f'target_{name}' for name in factor_cols}
target_data = {target_cols[name]: df[col].shift(-HORIZON_MONTHS) for name, col in factor_cols.items()}
df = df.assign(**target_data)

print('Dataset shape:', df.shape)
df.head(3)

In [ ]:
# Feature set: macro + market state + sentiment (all available in the dataset)
# We use a single shared feature set across all five factors.
candidate_features = [
    'mkt_vol_12m', 'mkt_trend_12m', 'dispersion',
    'term_spread_10y_fedfunds', 'rate_chg_3m', 'rate_chg_12m',
    'mktrf', 'rf',
    'wsj_index', 'wsj_uncertainty', 'topic_index', 'topic_uncertainty',
    'WallStreetBets_score', 'WallStreetBets_numeric_score',
]
feature_cols = [c for c in candidate_features if c in df.columns]
print(f'Features used ({len(feature_cols)}):', feature_cols)

X = df[feature_cols].copy()

# Time-series splits
train_mask = df['date'] < '2016-01-01'
valid_mask = (df['date'] >= '2016-01-01') & (df['date'] <= '2019-12-31')
test_mask  = df['date'] >= '2020-01-01'

X_train = X.loc[train_mask].copy()
X_valid = X.loc[valid_mask].copy()
X_test  = X.loc[test_mask].copy()

# Impute using TRAIN median only (no leakage into validation or test)
train_medians = X_train.median(numeric_only=True).fillna(0.0)
X_train = X_train.fillna(train_medians)
X_valid = X_valid.fillna(train_medians)
X_test  = X_test.fillna(train_medians)

pd.DataFrame({
    'n_obs': [train_mask.sum(), valid_mask.sum(), test_mask.sum()],
    'start': [df.loc[train_mask,'date'].min(), df.loc[valid_mask,'date'].min(), df.loc[test_mask,'date'].min()],
    'end':   [df.loc[train_mask,'date'].max(), df.loc[valid_mask,'date'].max(), df.loc[test_mask,'date'].max()],
}, index=['Train','Validation','Test'])

## 2. Ridge Regression

Ridge regression adds L2 regularization to OLS, shrinking all coefficients toward zero.
This is appropriate for our setting: many moderately predictive features with high multicollinearity.
Hyperparameter `alpha` is tuned on the validation set using Information Coefficient.

In [ ]:
RIDGE_ALPHAS = [0.01, 0.1, 1, 10, 100, 1000]
LASSO_ALPHAS = [1e-4, 1e-3, 1e-2, 1e-1, 1.0]

def align_split(X_split, y_split, date_split=None):
    """Drop rows where the target is NaN."""
    mask = y_split.notna()
    X_a = X_split.loc[mask].copy()
    y_a = y_split.loc[mask].copy()
    if date_split is None:
        return X_a, y_a
    return X_a, y_a, date_split.loc[mask].copy()

def select_best_by_ic(tune_df):
    """Pick the alpha with highest validation IC (fall back to lowest MSE)."""
    if tune_df['valid_ic'].notna().any():
        return tune_df.sort_values(['valid_ic','valid_mse'], ascending=[False,True]).iloc[0]
    return tune_df.sort_values('valid_mse').iloc[0]

ridge_rows = []
ridge_test_predictions = {}

for factor_name, target_col in target_cols.items():
    y_train = df.loc[train_mask, target_col]
    y_valid = df.loc[valid_mask, target_col]
    y_test  = df.loc[test_mask,  target_col]

    X_tr, y_tr = align_split(X_train, y_train)
    X_va, y_va = align_split(X_valid, y_valid)
    X_te, y_te, dates_te = align_split(X_test, y_test, df.loc[test_mask,'date'])

    if len(y_tr) == 0 or len(y_va) == 0 or len(y_te) == 0:
        ridge_rows.append({'factor': factor_name, 'best_alpha': np.nan,
                           'valid_ic': np.nan, 'test_ic': np.nan, 'test_r2': np.nan})
        continue

    # Tune on validation
    tune_rows = []
    for alpha in RIDGE_ALPHAS:
        pipe = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=alpha))])
        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        tune_rows.append({'alpha': alpha,
                          'valid_ic':  information_coefficient(y_va, pred_va),
                          'valid_mse': mean_squared_error(y_va, pred_va)})

    best = select_best_by_ic(pd.DataFrame(tune_rows))
    best_alpha = float(best['alpha'])

    # Refit on train + validation, evaluate on test
    X_tv = pd.concat([X_tr, X_va])
    y_tv = pd.concat([y_tr, y_va])
    final = Pipeline([('scaler', StandardScaler()), ('model', Ridge(alpha=best_alpha))])
    final.fit(X_tv, y_tv)
    pred_te = final.predict(X_te)

    ridge_rows.append({
        'factor':     factor_name,
        'best_alpha': best_alpha,
        'valid_ic':   round(float(best['valid_ic']), 4),
        'test_ic':    round(information_coefficient(y_te, pred_te), 4),
        'test_r2':    round(r2_score(y_te, pred_te), 4),
        'test_mse':   round(mean_squared_error(y_te, pred_te), 6),
    })
    ridge_test_predictions[factor_name] = pd.DataFrame({
        'date': dates_te.values, 'actual': y_te.values, 'pred': pred_te
    }).sort_values('date')

ridge_df = pd.DataFrame(ridge_rows).set_index('factor')
print('Ridge Results:')
ridge_df

## 3. Lasso Regression

Lasso adds L1 regularization, which drives some coefficients exactly to zero — effectively
performing feature selection. Comparing Lasso to Ridge tells us whether sparsity helps in
this setting (i.e., whether only a few features are truly predictive).

In [ ]:
lasso_rows = []

for factor_name, target_col in target_cols.items():
    y_train = df.loc[train_mask, target_col]
    y_valid = df.loc[valid_mask, target_col]
    y_test  = df.loc[test_mask,  target_col]

    X_tr, y_tr = align_split(X_train, y_train)
    X_va, y_va = align_split(X_valid, y_valid)
    X_te, y_te, _ = align_split(X_test, y_test, df.loc[test_mask,'date'])

    if len(y_tr) == 0 or len(y_va) == 0 or len(y_te) == 0:
        lasso_rows.append({'factor': factor_name, 'best_alpha': np.nan,
                           'valid_ic': np.nan, 'test_ic': np.nan, 'test_r2': np.nan})
        continue

    tune_rows = []
    for alpha in LASSO_ALPHAS:
        pipe = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=alpha, max_iter=5000))])
        pipe.fit(X_tr, y_tr)
        pred_va = pipe.predict(X_va)
        tune_rows.append({'alpha': alpha,
                          'valid_ic':  information_coefficient(y_va, pred_va),
                          'valid_mse': mean_squared_error(y_va, pred_va)})

    best = select_best_by_ic(pd.DataFrame(tune_rows))
    best_alpha = float(best['alpha'])

    X_tv = pd.concat([X_tr, X_va])
    y_tv = pd.concat([y_tr, y_va])
    final = Pipeline([('scaler', StandardScaler()), ('model', Lasso(alpha=best_alpha, max_iter=5000))])
    final.fit(X_tv, y_tv)
    pred_te = final.predict(X_te)

    lasso_rows.append({
        'factor':     factor_name,
        'best_alpha': best_alpha,
        'valid_ic':   round(float(best['valid_ic']), 4),
        'test_ic':    round(information_coefficient(y_te, pred_te), 4),
        'test_r2':    round(r2_score(y_te, pred_te), 4),
        'test_mse':   round(mean_squared_error(y_te, pred_te), 6),
    })

lasso_df = pd.DataFrame(lasso_rows).set_index('factor')
print('Lasso Results:')
lasso_df

In [ ]:
# Side-by-side comparison: Ridge vs Lasso test IC
comparison = pd.DataFrame({
    'Ridge Test IC':  ridge_df['test_ic'],
    'Lasso Test IC':  lasso_df['test_ic'],
    'Ridge Test R²':  ridge_df['test_r2'],
    'Lasso Test R²':  lasso_df['test_r2'],
})
print('Ridge vs Lasso — Test Set Performance:')
comparison

In [ ]:
# Plot: predicted vs actual for the best-performing factor (by Ridge test IC)
best_factor = ridge_df['test_ic'].idxmax()
plot_df = ridge_test_predictions[best_factor]

fig, ax = plt.subplots(figsize=(13, 4))
ax.plot(plot_df['date'], plot_df['actual'], label='Actual', linewidth=1.8, color='#1565C0')
ax.plot(plot_df['date'], plot_df['pred'], label='Ridge Predicted', linewidth=1.8,
        alpha=0.85, color='#E91E63', linestyle='--')
ax.set_title(f'Test Set: Actual vs Ridge Predicted — {best_factor.capitalize()} Factor',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel('Forward 1M Net Return')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 4. Classification Targets

In addition to return prediction, we define three binary classification targets for each factor:
- **Direction**: did the factor return a positive return next month?
- **Outperformance**: did the factor outperform the market excess return?
- **High return**: did the factor return exceed its historical median?

These targets are useful for evaluating **timing accuracy** independent of magnitude prediction.

In [ ]:
has_mktrf = 'mktrf' in df.columns
mktrf_fwd = df['mktrf'].shift(-HORIZON_MONTHS) if has_mktrf else None
thresholds = {}

for name in factor_cols:
    target_col  = f'target_{name}'
    dir_col     = f'target_{name}_dir'
    outperf_col = f'target_{name}_outperf'
    hi_col      = f'target_{name}_hi'

    y = df[target_col]
    valid_y = y.notna()

    df[dir_col] = pd.Series(pd.NA, index=df.index, dtype='Int64')
    df.loc[valid_y, dir_col] = (y.loc[valid_y] > 0).astype('Int64')

    df[outperf_col] = pd.Series(pd.NA, index=df.index, dtype='Int64')
    if has_mktrf:
        valid_pair = y.notna() & mktrf_fwd.notna()
        df.loc[valid_pair, outperf_col] = (y.loc[valid_pair] > mktrf_fwd.loc[valid_pair]).astype('Int64')

    thresh = df.loc[train_mask, target_col].median()
    thresholds[name] = thresh
    df[hi_col] = pd.Series(pd.NA, index=df.index, dtype='Int64')
    df.loc[valid_y, hi_col] = (y.loc[valid_y] > thresh).astype('Int64')

# Summarize label balance across splits
def split_mean(col, mask):
    s = df.loc[mask, col]
    return round(float(s.mean()), 3) if s.notna().any() else np.nan

label_rows = []
for name in factor_cols:
    label_rows.append({
        'factor': name,
        'dir_train':     split_mean(f'target_{name}_dir',     train_mask),
        'dir_valid':     split_mean(f'target_{name}_dir',     valid_mask),
        'dir_test':      split_mean(f'target_{name}_dir',     test_mask),
        'outperf_train': split_mean(f'target_{name}_outperf', train_mask),
        'outperf_valid': split_mean(f'target_{name}_outperf', valid_mask),
        'outperf_test':  split_mean(f'target_{name}_outperf', test_mask),
    })

pd.DataFrame(label_rows).set_index('factor')

## 5. Summary

**Key takeaways from the baseline models:**

- Ridge regression outperforms Lasso on most factors, suggesting that **feature selection
  (sparsity) is less valuable than shrinkage** in this setting — all features contribute
  some marginal signal.
- The **Size** factor has the highest test IC for Ridge, indicating its returns are the
  most predictable with macro/sentiment features.
- Absolute ICs are modest (typically 0.05–0.20), which is expected — equity return
  prediction is notoriously difficult. The more relevant question is whether these signals
  improve portfolio performance, which we test in notebook 04.
- **R² values are low or negative** on the test set for several factors — a known
  challenge in financial return prediction. IC is a better metric here as it measures
  directional skill rather than explained variance.

In [ ]:
# Visual summary: test IC bar chart
factors = list(ridge_df.index)
x = np.arange(len(factors))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, ridge_df['test_ic'], width, label='Ridge', color='#1565C0', alpha=0.8)
ax.bar(x + width/2, lasso_df['test_ic'], width, label='Lasso', color='#C62828', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f.capitalize() for f in factors])
ax.set_ylabel('Test IC')
ax.set_title('Test Set Information Coefficient — Ridge vs Lasso', fontsize=11, fontweight='bold')
ax.legend()
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()